### 분석 라이브러리 로드 및 사용 준비하기

In [1]:
import geopandas as gpd # 지리정보 파일 읽고 처리 # Anaconda Prompt에서 pip install geopandas로 설치해야 함
import pandas as pd # 표 형식 데이터 처리
import matplotlib.pyplot as plt # 그래프, 지도 등 시각화

In [2]:
plt.rcParams['font.family'] = 'Malgun Gothic' # 윈도우에서 한글 폰트 설정

### 부산시 경계, 격자 경계 및 통계 데이터 불러오기

In [4]:
prj_dir = 'C:/Users/user/Desktop/sgis-data-manual/공간분석 사례 Case Studies/Python/부산시 인구 격자별 분석' # 'C:/SGIS/Python/부산시 인구 격자별 분석'
bord_sido = gpd.read_file(prj_dir + '/' + 'bnd_sido_21_2025_2Q.shp') # gpd.read_file(prj_dir + '/' + 'bnd_sido_21_2024_2Q.shp')
grid_mara = gpd.read_file(prj_dir + '/' + 'grid_마라_1K.shp')
grid_mama = gpd.read_file(prj_dir + '/' + 'grid_마마_1K.shp')
stat_mara = pd.read_csv(prj_dir + '/' + '2024년_인구_마라_1K.csv', encoding='CP949', # pd.read_csv(prj_dir + '/' + '2023년_인구_마라_1K.csv', encoding='CP949',
                   names=['BASE_YEAR', 'GRID_1K_CD', 'STAT_CD', 'POP'])
stat_mama = pd.read_csv(prj_dir + '/' + '2024년_인구_마마_1K.csv', encoding='CP949', # pd.read_csv(prj_dir + '/' + '2023년_인구_마마_1K.csv', encoding='CP949',
                   names=['BASE_YEAR', 'GRID_1K_CD', 'STAT_CD', 'POP'])

### 격자 통계 및 통계 데이터 병합하기

In [5]:
grid_merged = pd.concat([grid_mara, grid_mama], ignore_index=True)
stat_merged = pd.concat([stat_mara, stat_mama], ignore_index=True)

### 데이터 구조 확인하기

In [6]:
print(grid_merged.info()) # 격자 경계 코드(GRID_1K_CD), 지리정보(geometry)로 구성
print(stat_merged.info()) # 기준날짜(BASE_YEAR), 격자 경계 코드(GRID_1K_CD), 통계 코드(STAT_CD), 인구수(POP)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   GRID_CD   9551 non-null   object  
 1   geometry  9551 non-null   geometry
dtypes: geometry(1), object(1)
memory usage: 149.4+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260662 entries, 0 to 260661
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   BASE_YEAR   260662 non-null  int64 
 1   GRID_1K_CD  260662 non-null  object
 2   STAT_CD     260662 non-null  object
 3   POP         260662 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 8.0+ MB
None


### 통계 항목 필터링

In [7]:
# STAT_CD 열에 있는 고유 값 확인
unique_stat_cd = stat_merged['STAT_CD'].unique()
print(unique_stat_cd)

['in_age_005' 'in_age_006' 'in_age_007' 'in_age_008' 'in_age_011'
 'in_age_012' 'in_age_013' 'in_age_014' 'in_age_015' 'in_age_016'
 'in_age_017' 'in_age_035' 'in_age_036' 'in_age_037' 'in_age_041'
 'in_age_042' 'in_age_043' 'in_age_044' 'in_age_045' 'in_age_046'
 'in_age_047' 'in_age_066' 'in_age_068' 'in_age_071' 'in_age_073'
 'in_age_074' 'in_age_075' 'in_age_076' 'in_age_077' 'to_in_001'
 'to_in_007' 'to_in_008' 'in_age_003' 'in_age_004' 'in_age_009'
 'in_age_010' 'in_age_018' 'in_age_019' 'in_age_020' 'in_age_034'
 'in_age_038' 'in_age_039' 'in_age_040' 'in_age_048' 'in_age_063'
 'in_age_064' 'in_age_070' 'in_age_072' 'in_age_078' 'in_age_079'
 'in_age_080' 'in_age_033' 'in_age_065' 'in_age_069' 'in_age_002'
 'in_age_062' 'in_age_067' 'in_age_032' 'in_age_049' 'in_age_001'
 'in_age_061' 'in_age_031' 'in_age_050' 'in_age_021' 'in_age_081'
 'in_age_051']


In [8]:
# 총인구('to_in_001') 데이터만 필터링
stat_total = stat_merged[stat_merged['STAT_CD'] == 'to_in_001']

# 필터링 후 STAT_CD 고유값 재확인
unique_stat_cd = stat_total['STAT_CD'].unique()
print(unique_stat_cd)

['to_in_001']


### 분석에 필요한 컬럼만 선택하기

In [10]:
stat_total = stat_total[['GRID_1K_CD', 'POP']]

### 부산시 경계와 격자 경계가 교차하는 격자만 선택하기

In [13]:
grid_intersects = grid_merged[grid_merged.geometry.intersects(bord_sido.geometry.union_all())]

### 경계와 통계 병합하기

In [15]:
grid_intersects = grid_intersects.merge(stat_total, on='GRID_1K_CD', how='left')
# 결합 후 통계값이 없는 구간에 NaN이 들어가 있을 수 있음(결측값 확인)
print(grid_intersects.isna().sum())

KeyError: 'GRID_1K_CD'